# 09 - Evidence-Grounded Hypothesis Generation

**Objective**: Formulate formal deductive scientific hypotheses grounded in retrieved literature passages. Condition Large Language Models (LLM) on empirical evidence pools, target populations, and research objectives to synthesize testable mHealth hypotheses.

**Inputs**:
- `outputs/candidate_research_gaps.csv` (Mined literature cavities)
- `outputs/retrieved_gap_evidence.json` (Passage pools)
- User theme & target population constraints

**Outputs**:
- `outputs/generated_hypotheses.json` (Structured hypothesis objects)
- Provenance linkages connecting generated propositions to cited PMIDs

**Production Code Reused**:
- `src/phase2/hypothesis_generation.py`


In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.phase2.hypothesis_generation import HypothesisGenerator, ScientificHypothesis

print("Environment ready.")


## 1. Inspect the Generation Engine & Prompt Blueprint
Hypothesis generation enforces a structured scientific contract:
- Formal hypothesis statement
- Independent Variable (IV) & Dependent Variable (DV)
- Target population specification
- Proposed causal/functional mechanism
- Cited evidence IDs (must match retrieved passages)


In [ ]:
# Initialize generator (uses mock or live Gemini depending on GEMINI_API_KEY)
api_key = os.environ.get("GEMINI_API_KEY")
use_mock = api_key is None

generator = HypothesisGenerator(use_mock=use_mock, api_key=api_key)
print(f"Generator Mode: {'Live API (Google Gemini)' if not generator.use_mock else 'Deterministic Mock / Verified Offline'}")
print(f"Configured Model: {generator.model_name}")


## 2. Inspect Verified Generated Hypotheses
We load the verified generation output from `outputs/generated_hypotheses.json` to inspect the generated scientific propositions.


In [ ]:
with open("outputs/generated_hypotheses.json", "r", encoding="utf-8") as f:
    hyps_data = json.load(f)

print(f"Total Verified Hypotheses: {len(hyps_data)}")

for idx, h in enumerate(hyps_data[:3], 1):
    print(f"\n{'='*75}")
    print(f"HYPOTHESIS #{idx}: {h.get('hypothesis_id')} (Linked Gap: {h.get('gap_id')})")
    print(f"{'='*75}")
    print(f"Statement: \"{h.get('hypothesis')}\"")
    print(f"Target Cohort: {h.get('target_population')}")
    print(f"Independent Variable: {h.get('independent_variable')}")
    print(f"Dependent Variable:   {h.get('dependent_variable')}")
    print(f"Proposed Mechanism:   {h.get('proposed_mechanism')}")
    print(f"Directionality:       {h.get('directionality')}")
    print(f"Cited Evidence IDs:   {h.get('supporting_evidence_ids')}")


## 3. Live Single-Prompt Demonstration
We demonstrate prompt assembly with retrieved context for a specific clinical inquiry.


In [ ]:
sample_gap = {
    "gap_id": "GAP-001",
    "concept_a": "health services accessibility",
    "concept_b": "rural health services",
    "gap_score": 1.1809
}

evidence_passages = [
    {
        "chunk_id": "CHK-001",
        "pmid": "31456789",
        "text": "Asynchronous store-and-forward tele-dermatology reduced wait times by 48% across 12 rural clinics in Western Australia.",
        "similarity_score": 0.842
    },
    {
        "chunk_id": "CHK-002",
        "pmid": "32987654",
        "text": "Cellular connectivity limitations in mountainous districts resulted in a 31% packet drop rate for real-time video consults.",
        "similarity_score": 0.791
    }
]

prompt = generator._build_prompt(
    gap_record=sample_gap,
    evidence_chunks=evidence_passages,
    n_hypotheses=1,
    user_theme="Telemedicine triage in remote communities",
    target_population="Rural elderly patients with mobility constraints",
    research_objective="Reduce emergency clinic travel time"
)

print("Constructed Scientific Prompt Preview (First 800 chars):")
print("-" * 60)
print(prompt[:800] + "\n...")
